In [ ]:
import os 
import sqlite3
import pandas as pd 
import numpy as np
import tensorflow as tf
import h5py
import keras 
from keras import layers, models, Input
from keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


# make it into the correct shape
def paddington(data,padVal, div=100):
    
    reminder = len(data)%div

    if reminder == 0:
        return data

    amount_to_pad = div-reminder

    if data.ndim == 1:
        padded = np.pad(data,(0,amount_to_pad),constant_values=padVal)
    else:
        padded = np.pad(data, ((0, amount_to_pad), (0, 0)), constant_values=padVal) 
    return padded       

"""
This function takes one of our two data sources, CSV and SQLite, which is a direct replica of the CSV file
and it checks whether they produce similar accuracy for key and quality in training as well as on the test split
it also takes source_name: which can be just a String, not exact match, useful for the print only
"""

def evaluate_csv_sqlite_matching(df_data, source_name):
    x_data = df_data.drop(columns=['key','quality'])
    y_key_data = df_data['key']
    y_quality_data = df_data['quality']
    
    # split into train validate and test
    x_train, x_test, y_key_train, y_key_test, y_quality_train, y_quality_test = train_test_split(x_data,
                                                                                                 y_key_data,
                                                                                                 y_quality_data,
                                                                                                 test_size=0.2,
                                                                                                 random_state=42)
    # add padding 
    padding = 0
    x_test_pad = paddington(x_test,padding)
    x_train_pad = paddington(x_train,padding)
    y_key_test = paddington(y_key_test,padding)
    y_key_train = paddington(y_key_train,padding)
    y_quality_test = paddington(y_quality_test,padding)
    y_quality_train = paddington(y_quality_train,padding)

    x_train_batches = x_train_pad.reshape(-1, 100, 25)
    x_test_batches  = x_test_pad.reshape(-1, 100, 25)
    y_key_train_batches = y_key_train.reshape(-1,100)
    y_quality_train_batches = y_quality_train.reshape(-1,100)
    y_key_test_batches = y_key_test.reshape(-1, 100)
    y_quality_test_batches = y_quality_test.reshape(-1, 100)

    inputs = Input(shape=(100,25))
    output = layers.Conv1D(filters=24, kernel_size=3,strides=1,padding="same",activation='relu')(inputs)
    output = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(output)

    keyInput= layers.Dense(32,activation='relu')(output)
    key= layers.Dense(14,activation= 'softmax',name='key')(keyInput)

    shapeInput = layers.Dense(32,activation='relu')(output)
    quality = layers.Dense(4,activation='softmax',name='quality')(shapeInput)

    model = models.Model(inputs=inputs, outputs = [key, quality])
    model.compile(optimizer='adam',
              loss= 'sparse_categorical_crossentropy',
              loss_weights={'key': 1.0, 'quality': 1.0},
              metrics=['accuracy'])

# to keep the key and quality as important, maybe try maybe try to
# make sure both key and quality outputs result equally in the total loss
# loss_weights=? 

    # training the model
    model.fit(x_train_batches, {'key': y_key_train_batches, 'quality': y_quality_train_batches}, 
                      batch_size=32, 
                      epochs=5)

    # evaluating our test split 0.2
    test_batch_results = model.evaluate(x_test_batches, {'key': y_key_test_batches, 'quality': y_quality_test_batches}, 
                      batch_size=32)
    
    key_accuracy = test_batch_results[3]
    quality_accuracy = test_batch_results[4]
    
    print(f"\n\nThe results using the {source_name} are:\n")
    print(f" - Key accuracy: {key_accuracy}")
    print(f" - Quality accuracy: {quality_accuracy}")
    
    return key_accuracy, quality_accuracy

# Evaluating results using CSV vs SQLite
sourceName1 = 'CSV Data'
df_data = pd.read_csv(filepath_or_buffer='cleaned_data.csv')
evaluate_csv_sqlite_matching(df_data, sourceName1)

sourceName2 = 'SQLite Data'
conn = sqlite3.connect('cleaned_data.db') # name of sqlite db you gave 
# make sure to use the name of the table that you are using in the sqlite db 
df_data = pd.read_sql_query("SELECT * FROM training_data_v0", conn)
conn.close()
evaluate_csv_sqlite_matching(df_data, sourceName2)


